In [1]:
# ============================================================
# CLEAN AND COMBINE THE 9 DATASETS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import re
from io import StringIO

In [2]:
# ============================================================
# STEP 1. SET YOUR DATA FOLDER
# ============================================================

# Change this to the folder containing your 9 CSV files
DATA_DIR = Path(".")

# Define file paths

files = {
    "CPI": DATA_DIR / "CPIAUCSL.csv",
    "VIX": DATA_DIR / "VIXCLS.csv",
    "Baa–10Y credit spread": DATA_DIR / "BAA10Y.csv",
    "10Y–2Y term spread": DATA_DIR / "T10Y2Y.csv",
    "Unemployment rate": DATA_DIR / "UNRATE.csv",
    "2-year Treasury yield": DATA_DIR / "DGS2.csv",
    "Industrial Production": DATA_DIR / "INDPRO.csv",
    "Momentum factor": DATA_DIR / "F-F_Momentum_Factor.csv",
    "MKT-RF, SMB, HML, RMW, CMA, RF": DATA_DIR / "F-F_Research_Data_5_Factors_2x3.csv"
}

for name, path in files.items():
    print(f"{name}: {path.name} | Exists: {path.exists()}")

OUTPUT_FILE = DATA_DIR / "combined_clean_data.csv"


CPI: CPIAUCSL.csv | Exists: True
VIX: VIXCLS.csv | Exists: True
Baa–10Y credit spread: BAA10Y.csv | Exists: True
10Y–2Y term spread: T10Y2Y.csv | Exists: True
Unemployment rate: UNRATE.csv | Exists: True
2-year Treasury yield: DGS2.csv | Exists: True
Industrial Production: INDPRO.csv | Exists: True
Momentum factor: F-F_Momentum_Factor.csv | Exists: True
MKT-RF, SMB, HML, RMW, CMA, RF: F-F_Research_Data_5_Factors_2x3.csv | Exists: True


In [3]:
# ============================================================
# STEP 2. HELPER FUNCTION FOR FRED DATA
# ============================================================

def clean_fred(filename, variable_name, monthly_method="mean"):
    """
    Read a FRED CSV file, clean missing observations,
    and convert it to monthly frequency.
    """

    path = DATA_DIR / filename

    df = pd.read_csv(path)

    # Clean column names
    df.columns = df.columns.str.strip()

    # FRED normally uses DATE as first column
    date_col = df.columns[0]
    value_col = df.columns[1]

    # Convert date
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    # Convert values to numeric
    # FRED sometimes uses "." for missing values
    df[value_col] = pd.to_numeric(
        df[value_col].replace(".", np.nan),
        errors="coerce"
    )

    # Remove invalid rows
    df = df.dropna(subset=[date_col])

    # Rename
    df = df.rename(
        columns={
            date_col: "Date",
            value_col: variable_name
        }
    )

    # Set date as index
    df = df.set_index("Date").sort_index()

    # Convert to monthly
    if monthly_method == "mean":
        df = df.resample("ME").mean()

    elif monthly_method == "last":
        df = df.resample("ME").last()

    else:
        raise ValueError("monthly_method must be 'mean' or 'last'")

    return df

# ============================================================
# STEP 3. CLEAN VIX
# ============================================================

vix = clean_fred(
    "VIXCLS.csv",
    "VIX",
    monthly_method="mean"
)


# ============================================================
# STEP 4. CLEAN CPI
# ============================================================

cpi = clean_fred(
    "CPIAUCSL.csv",
    "CPI",
    monthly_method="last"
)

# Monthly CPI
cpi = cpi.dropna(subset=["CPI"])

cpi["Inflation"] = (
    cpi["CPI"]
    .pct_change(fill_method=None)
    * 100
)

# ============================================================
# STEP 5. CLEAN BAA CREDIT SPREAD
# ============================================================

baa = clean_fred(
    "BAA10Y.csv",
    "Credit_Spread",
    monthly_method="mean"
)


# ============================================================
# STEP 6. CLEAN TERM SPREAD
# ============================================================

term = clean_fred(
    "T10Y2Y.csv",
    "Term_Spread",
    monthly_method="mean"
)


# ============================================================
# STEP 7. CLEAN UNEMPLOYMENT RATE
# ============================================================

unrate = clean_fred(
    "UNRATE.csv",
    "Unemployment",
    monthly_method="last"
)


# ============================================================
# STEP 8. CLEAN 2-YEAR TREASURY YIELD
# ============================================================

dgs2 = clean_fred(
    "DGS2.csv",
    "Treasury_2Y",
    monthly_method="mean"
)


# ============================================================
# STEP 9. CLEAN INDUSTRIAL PRODUCTION
# ============================================================

indpro = clean_fred(
    "INDPRO.csv",
    "Industrial_Production",
    monthly_method="last"
)

# Monthly industrial production growth
indpro = indpro.dropna(
    subset=["Industrial_Production"]
)

indpro["IP_Growth"] = (
    indpro["Industrial_Production"]
    .pct_change(fill_method=None)
    * 100
)

In [4]:
# ============================================================
# FUNCTION TO CLEAN KENNETH FRENCH FILES
# ============================================================

def clean_french_file(filename):
    """
    Clean Kenneth French monthly factor CSV files.

    Handles:
    - metadata rows before the data
    - YYYYMM dates
    - YYYY-MM dates
    - YYYY/MM dates
    - different numbers of columns
    """

    path = DATA_DIR / filename

    # --------------------------------------------------------
    # STEP 1. READ FILE AS PLAIN TEXT
    # --------------------------------------------------------

    with open(path, "r", encoding="utf-8-sig") as f:
        lines = f.readlines()

    # --------------------------------------------------------
    # STEP 2. FIND MONTHLY DATA ROWS
    # --------------------------------------------------------

    first_data_line = None

    for i, line in enumerate(lines):

        first_item = line.split(",")[0].strip()

        # Accept:
        # 192701
        # 1927-01
        # 1927/01

        if (
            re.fullmatch(r"\d{6}", first_item)
            or re.fullmatch(r"\d{4}-\d{2}", first_item)
            or re.fullmatch(r"\d{4}/\d{2}", first_item)
        ):
            first_data_line = i
            break

    if first_data_line is None:
        raise ValueError(
            f"Could not find monthly observations in {filename}"
        )

    # --------------------------------------------------------
    # STEP 3. FIND HEADER
    # --------------------------------------------------------

    header_line = first_data_line - 1

    print(
        f"\n{filename}"
    )

    print(
        "Detected header:",
        lines[header_line].strip()
    )

    # --------------------------------------------------------
    # STEP 4. EXTRACT ONLY MONTHLY SECTION
    # --------------------------------------------------------

    monthly_lines = []

    # Add header first
    monthly_lines.append(
        lines[header_line]
    )

    for line in lines[first_data_line:]:

        first_item = line.split(",")[0].strip()

        if (
            re.fullmatch(r"\d{6}", first_item)
            or re.fullmatch(r"\d{4}-\d{2}", first_item)
            or re.fullmatch(r"\d{4}/\d{2}", first_item)
        ):

            monthly_lines.append(line)

        else:

            # Once monthly data has started,
            # stop when another section begins
            if len(monthly_lines) > 1:
                break

    # --------------------------------------------------------
    # STEP 5. CONVERT EXTRACTED TEXT TO DATAFRAME
    # --------------------------------------------------------

    text = "".join(monthly_lines)

    data = pd.read_csv(
        StringIO(text)
    )

    # --------------------------------------------------------
    # STEP 6. CLEAN COLUMN NAMES
    # --------------------------------------------------------

    data.columns = (
        data.columns
        .astype(str)
        .str.strip()
    )

    # First column may have blank / unnamed header
    data = data.rename(
        columns={
            data.columns[0]: "Date"
        }
    )

    # --------------------------------------------------------
    # STEP 7. CLEAN DATE
    # --------------------------------------------------------

    date_string = (
        data["Date"]
        .astype(str)
        .str.strip()
        .str.replace("-", "", regex=False)
        .str.replace("/", "", regex=False)
    )

    data["Date"] = pd.to_datetime(
        date_string,
        format="%Y%m",
        errors="coerce"
    )

    # Convert to month-end
    data["Date"] = (
        data["Date"]
        + pd.offsets.MonthEnd(0)
    )

    # --------------------------------------------------------
    # STEP 8. CONVERT FACTOR COLUMNS TO NUMERIC
    # --------------------------------------------------------

    for col in data.columns[1:]:

        data[col] = (
            data[col]
            .astype(str)
            .str.strip()
        )

        data[col] = pd.to_numeric(
            data[col],
            errors="coerce"
        )

        # Kenneth French missing codes
        data[col] = data[col].replace(
            [-99.99, -999, -999.0],
            np.nan
        )

    # --------------------------------------------------------
    # STEP 9. REMOVE INVALID ROWS
    # --------------------------------------------------------

    data = data.dropna(
        subset=["Date"]
    )

    data = data.dropna(
        axis=1,
        how="all"
    )

    # --------------------------------------------------------
    # STEP 10. SET DATE INDEX
    # --------------------------------------------------------

    data = (
        data
        .set_index("Date")
        .sort_index()
    )

    return data

# ============================================================
# STEP 11. CLEAN MOMENTUM FACTOR
# ============================================================

momentum = clean_french_file(
    "F-F_Momentum_Factor.csv"
)

print("\nMomentum columns:")
print(momentum.columns)

print("\nFirst 5 rows:")
print(momentum.head())


# Rename momentum factor
if "Mom" in momentum.columns:
    momentum = momentum.rename(
        columns={"Mom": "MOM"}
    )

elif len(momentum.columns) == 1:
    momentum.columns = ["MOM"]


# ============================================================
# STEP 12. CLEAN FAMA-FRENCH 5 FACTORS
# ============================================================

ff5 = clean_french_file(
    "F-F_Research_Data_5_Factors_2x3.csv"
)

print("\nFama-French 5-factor columns:")
print(ff5.columns)

print("\nFirst 5 rows:")
print(ff5.head())


# Rename columns to easier Python names
ff5 = ff5.rename(
    columns={
        "Mkt-RF": "MKT_RF",
        "Mkt_RF": "MKT_RF",
        "SMB": "SMB",
        "HML": "HML",
        "RMW": "RMW",
        "CMA": "CMA",
        "RF": "RF"
    }
)


F-F_Momentum_Factor.csv
Detected header: ,Mom

Momentum columns:
Index(['Mom'], dtype='object')

First 5 rows:
             Mom
Date            
1927-01-31  0.57
1927-02-28 -1.51
1927-03-31  3.52
1927-04-30  4.35
1927-05-31  2.78

F-F_Research_Data_5_Factors_2x3.csv
Detected header: ,Mkt-RF,SMB,HML,RMW,CMA,RF

Fama-French 5-factor columns:
Index(['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF'], dtype='object')

First 5 rows:
            Mkt-RF   SMB   HML   RMW   CMA    RF
Date                                            
1963-07-31   -0.39 -0.48 -0.84  0.64 -1.15  0.27
1963-08-31    5.08 -0.80  1.72  0.40 -0.38  0.25
1963-09-30   -1.57 -0.43 -0.02 -0.78  0.15  0.27
1963-10-31    2.54 -1.34 -0.03  2.79 -2.25  0.29
1963-11-30   -0.86 -0.85  1.78 -0.43  2.27  0.27


In [5]:
# ============================================================
# STEP 13. STANDARDIZE COLUMN NAMES
# ============================================================

# Momentum
momentum = momentum.rename(
    columns={
        "Mom": "MOM"
    }
)

# Fama-French 5 factors
ff5 = ff5.rename(
    columns={
        "Mkt-RF": "MKT_RF"
    }
)

print("Momentum columns:", momentum.columns.tolist())
print("FF5 columns:", ff5.columns.tolist())

Momentum columns: ['MOM']
FF5 columns: ['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']


In [6]:
# ============================================================
# STEP 14. MERGE ALL 9 DATASETS
# ============================================================

combined = pd.concat(
    [
        vix,
        cpi,
        baa,
        term,
        unrate,
        dgs2,
        indpro,
        momentum,
        ff5
    ],
    axis=1,
    join="outer"
)

# Sort by date
combined = combined.sort_index()

# Remove completely empty rows
combined = combined.dropna(how="all")

print("All 9 datasets merged successfully.")

All 9 datasets merged successfully.


In [7]:
# ============================================================
# STEP 15. OVERVIEW OF FINAL COMBINED DATASET
# ============================================================

print("\n========================================")
print("FINAL COMBINED DATASET OVERVIEW")
print("========================================")

print("\n1. Shape:")
print(combined.shape)

print("\n2. Date range:")
print("Start:", combined.index.min())
print("End:  ", combined.index.max())

print("\n3. Columns:")
print(combined.columns.tolist())

print("\n4. First 10 rows:")
print(combined.head(10))

print("\n5. Last 10 rows:")
print(combined.tail(10))

print("\n6. Data types:")
print(combined.dtypes)

print("\n7. Missing values:")
print(combined.isna().sum())

print("\n8. Number of duplicated dates:")
print(combined.index.duplicated().sum())

print("\n9. Summary statistics:")
print(combined.describe().T)


FINAL COMBINED DATASET OVERVIEW

1. Shape:
(1293, 16)

2. Date range:
Start: 1919-01-31 00:00:00
End:   2026-09-30 00:00:00

3. Columns:
['VIX', 'CPI', 'Inflation', 'Credit_Spread', 'Term_Spread', 'Unemployment', 'Treasury_2Y', 'Industrial_Production', 'IP_Growth', 'MOM', 'MKT_RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']

4. First 10 rows:
            VIX  CPI  Inflation  Credit_Spread  Term_Spread  Unemployment  \
Date                                                                        
1919-01-31  NaN  NaN        NaN            NaN          NaN           NaN   
1919-02-28  NaN  NaN        NaN            NaN          NaN           NaN   
1919-03-31  NaN  NaN        NaN            NaN          NaN           NaN   
1919-04-30  NaN  NaN        NaN            NaN          NaN           NaN   
1919-05-31  NaN  NaN        NaN            NaN          NaN           NaN   
1919-06-30  NaN  NaN        NaN            NaN          NaN           NaN   
1919-07-31  NaN  NaN        NaN            NaN 

In [8]:
# ============================================================
# STEP 16. MISSING DATA SUMMARY
# ============================================================

missing_summary = pd.DataFrame({
    "Missing_Count": combined.isna().sum(),
    "Missing_Percent": combined.isna().mean() * 100
})

missing_summary["Missing_Percent"] = (
    missing_summary["Missing_Percent"].round(2)
)

missing_summary = missing_summary.sort_values(
    "Missing_Percent",
    ascending=False
)

print(missing_summary)

                       Missing_Count  Missing_Percent
VIX                              852            65.89
Credit_Spread                    804            62.18
Treasury_2Y                      689            53.29
Term_Spread                      689            53.29
HML                              536            41.45
RMW                              536            41.45
MKT_RF                           536            41.45
SMB                              536            41.45
CMA                              536            41.45
RF                               536            41.45
Unemployment                     350            27.07
Inflation                        340            26.30
CPI                              339            26.22
MOM                               98             7.58
IP_Growth                          3             0.23
Industrial_Production              2             0.15


In [9]:
# ============================================================
# STEP 17. CHECK START/END DATE OF EACH VARIABLE
# ============================================================

coverage = []

for col in combined.columns:

    valid = combined[col].dropna()

    coverage.append({
        "Variable": col,
        "Start_Date": valid.index.min(),
        "End_Date": valid.index.max(),
        "Observations": len(valid)
    })

coverage = pd.DataFrame(coverage)

print(coverage)

                 Variable Start_Date   End_Date  Observations
0                     VIX 1990-01-31 2026-09-30           441
1                     CPI 1947-01-31 2026-07-31           954
2               Inflation 1947-02-28 2026-07-31           953
3           Credit_Spread 1986-01-31 2026-09-30           489
4             Term_Spread 1976-06-30 2026-09-30           604
5            Unemployment 1948-01-31 2026-08-31           943
6             Treasury_2Y 1976-06-30 2026-09-30           604
7   Industrial_Production 1919-01-31 2026-07-31          1291
8               IP_Growth 1919-02-28 2026-07-31          1290
9                     MOM 1927-01-31 2026-07-31          1195
10                 MKT_RF 1963-07-31 2026-07-31           757
11                    SMB 1963-07-31 2026-07-31           757
12                    HML 1963-07-31 2026-07-31           757
13                    RMW 1963-07-31 2026-07-31           757
14                    CMA 1963-07-31 2026-07-31           757
15      

In [10]:
print(combined.index[:20])

DatetimeIndex(['1919-01-31', '1919-02-28', '1919-03-31', '1919-04-30',
               '1919-05-31', '1919-06-30', '1919-07-31', '1919-08-31',
               '1919-09-30', '1919-10-31', '1919-11-30', '1919-12-31',
               '1920-01-31', '1920-02-29', '1920-03-31', '1920-04-30',
               '1920-05-31', '1920-06-30', '1920-07-31', '1920-08-31'],
              dtype='datetime64[ns]', name='Date', freq='ME')


In [11]:
# ============================================================
# STEP 18. SAVE FULL CLEAN COMBINED DATASET
# ============================================================

combined.to_csv(
    "combined_9_datasets_clean.csv",
    index=True
)

print("Saved: combined_9_datasets_clean.csv")

Saved: combined_9_datasets_clean.csv


In [12]:
# ============================================================
# STEP 19. FIND COMMON DATA AVAILABILITY
# ============================================================

common_start = max(
    combined[col].first_valid_index()
    for col in combined.columns
    if combined[col].first_valid_index() is not None
)

common_end = min(
    combined[col].last_valid_index()
    for col in combined.columns
    if combined[col].last_valid_index() is not None
)

print("Common start date:", common_start)
print("Common end date:", common_end)

Common start date: 1990-01-31 00:00:00
Common end date: 2026-07-31 00:00:00


In [13]:
research_data = combined.loc[
    common_start:common_end
].copy()

print("\nResearch dataset shape:")
print(research_data.shape)

print("\nMissing values:")
print(research_data.isna().sum())


Research dataset shape:
(439, 16)

Missing values:
VIX                      0
CPI                      1
Inflation                1
Credit_Spread            0
Term_Spread              0
Unemployment             1
Treasury_2Y              0
Industrial_Production    0
IP_Growth                0
MOM                      0
MKT_RF                   0
SMB                      0
HML                      0
RMW                      0
CMA                      0
RF                       0
dtype: int64
